# 09 — Final CNN V4 Evaluation and Release Decision

> This notebook summarises the completed V4 evaluation report. It reads the saved JSON and does not retrain or alter thresholds.


In [ ]:
from pathlib import Path
import json, pandas as pd

ROOT = Path.cwd()
report = json.loads((ROOT / 'documentation' / 'v4_training_report.json').read_text(encoding='utf-8'))
v4 = report['test_metrics_at_selected_threshold']
v3 = report['v3_baseline_on_same_v4_validation_and_test_splits']['test_metrics']
rows = []
for name, metrics in [('V3 historical baseline', v3), ('V4 active CNN', v4)]:
    rows.append({'Model': name, 'Threshold': metrics['threshold'], 'Accuracy': metrics['accuracy'], 'Balanced accuracy': metrics['balanced_accuracy'], 'AI precision': metrics['ai_precision'], 'AI recall': metrics['ai_recall'], 'AI F1': metrics['ai_f1'], 'ROC-AUC': metrics['roc_auc']})
comparison = pd.DataFrame(rows)
display(comparison.style.format({c:'{:.2%}' for c in comparison.columns if c not in ['Model','Threshold']}))


## V4 test confusion matrix

Rows are actual classes [real, AI-generated]; columns are predicted classes [real, AI-generated].

| | Predicted real | Predicted AI |
|---|---:|---:|
| Actual real | 87 | 7 |
| Actual AI | 5 | 98 |

V4 test support: 94 real and 103 AI-generated images. Threshold 0.50 was selected on validation balanced accuracy before the test set was evaluated.


In [ ]:
generator = report['new_generator_challenge_test']
generator_rows = []
for name, item in generator.items():
    generator_rows.append({'Generator': name, 'Images': item['n'], 'Detected as AI': item['ai_detected'], 'Detection rate': item['ai_detection_rate']})
display(pd.DataFrame(generator_rows))


## Interpretation and limitations

- V4 test accuracy is 93.91%, balanced accuracy 93.85%, AI precision 93.33%, AI recall 95.15%, AI F1 94.23% and ROC-AUC 98.89%.
- Compared with V3 on the same test images (using each model's validation-selected threshold), V4 has slightly higher accuracy, recall, F1 and ROC-AUC; V3 has higher AI precision. This is a modest trade-off, not an overwhelming improvement.
- The new-generator AI-only subset contains GPT 4, Gemini 3 and Meta AI 2 examples. These counts are too small for reliable generator-specific claims.
- All named generator families were represented in the V4 development/training pool; this is not a held-out-generator test.
- Source labels were inherited from user-supplied folders and not independently authenticated.
- The model's sigmoid score is uncalibrated and is not a probability of fraud or authenticity.
- V3 weights are not shipped in this repository; only historical metrics and its source SHA-256 are retained.
